# HVSMR-2.0 FOLD 0 MSPE-SwinUNETR — fine-tuning (frozen backbone) FOLD 2


In [ ]:
!python -c "import monai" || pip install -q "monai[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
!python -c "import wandb" || pip install -q wandb
%matplotlib inline

FOLD_NUMBER = 2

In [ ]:
import os
import sys

colab = True

if colab:
    from google.colab import drive, runtime, userdata
    drive.mount('/content/drive')
    data_dir = r"/content/drive/MyDrive/HVSMR-2.0/cropped"
    root_dir = r"/content/drive/MyDrive/RUNS/HVSMR-2.0/FINE_TUNING/"
    baseline_dir = r"/content/drive/MyDrive/RUNS/HVSMR-2.0/FULL_TRAINING/"
    mspe_code_dir = r"/content/drive/MyDrive/MSPE"
    num_workers = os.cpu_count() or 0
    batch_size = 1
    cache_rate = 1.0
else:
    data_dir = r""
    root_dir = r""
    baseline_dir = r""
    mspe_code_dir = r""
    num_workers = 0
    cache_rate = 0.0
    batch_size = 1

sys.path.append(mspe_code_dir)
os.makedirs(root_dir, exist_ok=True)

In [ ]:
import glob
import time
import copy
from abc import ABC, abstractmethod
from collections import Counter

import numpy as np
import monai
import wandb
import matplotlib.pyplot as plt

from monai.apps import CrossValidation
from monai.config import print_config
from monai.data import (
    CacheDataset, decollate_batch, DataLoader, list_data_collate,
)
from monai.networks.blocks import PatchEmbed
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.losses import TverskyLoss
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, CropForegroundd, ScaleIntensityRangePercentilesd,
    DivisiblePadd, RandFlipd, RandAffined, Rand3DElasticd,
    RandShiftIntensityd, RandScaleIntensityd, RandGaussianNoised,
    RandAdjustContrastd, AsDiscrete,
)
from monai.utils import first, set_determinism

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, LinearLR

from swin_mspe import (
    MSPEPatchEmbedSwinNaiveRouting,
    MSPEPatchEmbedSwinOverlapping,
    MSPEPatchEmbedSwinDilatingK3,
    mspe_swin_train_step,
    mspe_swin_forward,
    get_aspect_preserving_target_size,
    img_resize,
    label_resize,
    pi_resize,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 on Ampere
AMP_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"device: {device}, AMP dtype: {AMP_DTYPE}")
print_config()

In [ ]:
# Deterministic training for reproducibility
set_determinism(seed=0)
# torch.backends.cudnn.benchmark = True  # enable if input sizes are fixed

In [ ]:
train_images = sorted(glob.glob(os.path.join(data_dir, "*_cropped.nii.gz")))
train_labels = sorted(glob.glob(os.path.join(data_dir, "*_cropped_seg.nii.gz")))
train_endpoints = sorted(glob.glob(os.path.join(data_dir, "*_cropped_seg_endpoints.nii.gz")))

dataset_files = [
    {"image": i, "label": l, "endpoint": e}
    for i, l, e in zip(train_images, train_labels, train_endpoints)
]

print(f"cases: {len(dataset_files)}")
assert len(dataset_files) == 60, f"expected 60 HVSMR-2.0 scans, but found {len(dataset_files)}"

In [ ]:
AUGMENT = True  

deterministic_preproc = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    Spacingd(keys=["image", "label"], pixdim=(0.75, 0.75, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRangePercentilesd(keys=["image"], lower=5, upper=95, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label"]),
]

# Random augmentation (never cached)
augmentation = [
    RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.2),
    RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.2),
    RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.2),
    RandAffined(
        keys=["image", "label"], prob=0.2,
        rotate_range=(0.26, 0.26, 0.26),  
        scale_range=(0.1, 0.1, 0.1),
        mode=("bilinear", "nearest"),
        padding_mode="zeros",
    ),
    RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.05),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.2),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.2),
    RandAdjustContrastd(keys=["image"], gamma=(0.8, 1.2), prob=0.2),
]

#  every spatial dim is divisible by 32 
pad = [DivisiblePadd(keys=["image", "label"], k=32, mode="constant")]

train_transforms = Compose(deterministic_preproc + (augmentation if AUGMENT else []) + pad)

val_transforms = Compose([
    LoadImaged(keys=["image", "label", "endpoint"]),
    EnsureChannelFirstd(keys=["image", "label", "endpoint"]),
    Orientationd(keys=["image", "label", "endpoint"], axcodes="RAS"),
    CropForegroundd(keys=["image", "label", "endpoint"], source_key="image"),
    Spacingd(
        keys=["image", "label", "endpoint"],
        pixdim=(0.75, 0.75, 1.0),
        mode=("bilinear", "nearest", "nearest"),
    ),
    ScaleIntensityRangePercentilesd(keys=["image"], lower=5, upper=95, b_min=0.0, b_max=1.0, clip=True),
    EnsureTyped(keys=["image", "label", "endpoint"]),
    DivisiblePadd(keys=["image", "label", "endpoint"], k=32, mode="constant"),
])

In [ ]:
class CVDataset(ABC, CacheDataset):
    """Base class to generate cross-validation datasets dynamically."""

    def __init__(self, data, transform, cache_num=sys.maxsize, cache_rate=1.0, num_workers=4):
        data = self._split_datalist(datalist=data)
        CacheDataset.__init__(
            self, data, transform, cache_num=cache_num, cache_rate=cache_rate, num_workers=num_workers,
        )

    @abstractmethod
    def _split_datalist(self, datalist):
        raise NotImplementedError(f"Subclass {self.__class__.__name__} must implement this method.")


fold_number = FOLD_NUMBER
total_fold_number = 5
folds = list(range(total_fold_number))

cv_splitter = CrossValidation(
    dataset_cls=CVDataset,
    data=dataset_files,
    nfolds=total_fold_number,
    seed=12345,
)

# Training folds 
train_ds = cv_splitter.get_dataset(
    folds=folds[0:fold_number] + folds[fold_number + 1:],
    transform=train_transforms,
    cache_rate=cache_rate,
    num_workers=num_workers,
)
train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, collate_fn=list_data_collate,
)

# Held-out validation fold
val_ds = cv_splitter.get_dataset(
    folds=fold_number,
    transform=val_transforms,
    cache_rate=cache_rate,
    num_workers=num_workers,
)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=num_workers)

print(f"fold {fold_number}: train cases {len(train_ds)}, val cases {len(val_ds)}")

In [ ]:
# One train sample check
check_data = first(train_loader)
image, label = check_data["image"][0][0].cpu(), check_data["label"][0][0].cpu()
print(f"image shape: {image.shape}, label shape: {label.shape}")

D = min(80, image.shape[2] - 1)
plt.figure("check", (12, 6))
plt.subplot(1, 2, 1)
plt.title(f"train image (augment={AUGMENT})")
plt.imshow(image[:, :, D], cmap="gray")
plt.subplot(1, 2, 2)
plt.title("label")
plt.imshow(label[:, :, D])
plt.show()

In [ ]:
# Fine-tuning configuration (frozen backbone, MSPE-only)

FEATURE_SIZE = 48                
OUT_CHANNELS = 9                 
MAX_EPOCHS = 20
VAL_INTERVAL = 2
EARLY_STOPPING_PATIENCE = 100   

LR = 1e-3                        
WD = 1e-5
LOSS_ALPHA, LOSS_BETA = 0.3, 0.7  # Tversky FP/FN weighting

# MSPE 
MSPE_K = 3
MSPE_RESOLUTIONS = [96, 128, 192, 256]   # full-volume effective resolutions

SUBTRACT_ENDPOINTS = False       # optional endpoint-zone subtraction (kept off)

# Warm-start each MSPE kernel from the frozen baseline patch_embed 
WARM_START = True

DATASET_TAG = "HVSMR"
WANDB_PROJECT = "HVSMR-MSPE_SWIN_FINE_TUNING"

AUG_TAG = "AUG_1" if AUGMENT else "AUG_0"
AUG_SHORT = "A1" if AUGMENT else "A0"
FOLD_TAG = f"FOLD_{fold_number}"

# Frozen baseline (SwinUNETR V2, stock patch embed) trained by MSPE_SWIN_VOLUME_FOLD_0.ipynb
BASELINE_CHECKPOINT = os.path.join(
    baseline_dir, f"best_metric_BASELINE_SWIN_V2_fold{fold_number}_{AUG_SHORT}.pth"
)

# MSPE variants to fine-tune (frozen backbone, only patch_embed trains)
EXPERIMENTS = [
    {
        "name": "FT_ROUTING",
        "mspe_class": MSPEPatchEmbedSwinNaiveRouting,
        "description": "Frozen backbone + naive routing (K identical kernels)",
    },
    {
        "name": "FT_OVERLAPPING",
        "mspe_class": MSPEPatchEmbedSwinOverlapping,
        "description": "Frozen backbone + overlapping kernels",
    },
    {
        "name": "FT_DILATING",
        "mspe_class": MSPEPatchEmbedSwinDilatingK3,
        "description": "Frozen backbone + dilating kernels",
    },
]

for exp in EXPERIMENTS:
    exp["run_name"] = f"{exp['name']}_{FOLD_TAG}_{AUG_TAG}"
    exp["checkpoint_path"] = os.path.join(
        root_dir, f"best_metric_{exp['name']}_fold{fold_number}_{AUG_SHORT}.pth"
    )

print(f"Experiments to fine-tune: {[e['name'] for e in EXPERIMENTS]}")
print(f"Frozen baseline checkpoint: {BASELINE_CHECKPOINT}")
print(f"MSPE resolutions: {MSPE_RESOLUTIONS}, K: {MSPE_K}, warm-start kernels: {WARM_START}")
print(f"Epochs: {MAX_EPOCHS}, optimizer: AdamW, lr: {LR}, wd: {WD}, scheduler: CosineAnnealingLR")
print(f"Saving fine-tuned checkpoints to: {root_dir}")

In [ ]:
# Warm-start helpers + model factories (frozen backbone, MSPE-only)

# Set WANDB_API_KEY in your environment (or Colab secrets) before running
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

def login_wandb():
    if colab:
        wandb.login(key=WANDB_API_KEY)


def create_patch_embed_baseline():
    """Stock MONAI 3D patch embedding (matches the trained baseline)."""
    return PatchEmbed(
        patch_size=2, in_chans=1, embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm, spatial_dims=3,
    )


def _load_baseline_backbone_only(model, baseline_checkpoint):
    """Load every baseline weight EXCEPT swinViT.patch_embed.* (MSPE embed starts fresh)."""
    checkpoint = torch.load(baseline_checkpoint, weights_only=True)

    backbone_state = {
        name: tensor for name, tensor in checkpoint.items()
        if not name.startswith("swinViT.patch_embed.")
    }
    skipped_patch_keys = len(checkpoint) - len(backbone_state)

    incompatible = model.load_state_dict(backbone_state, strict=False)
    unexpected_keys = list(incompatible.unexpected_keys)
    missing_non_patch_keys = [
        name for name in incompatible.missing_keys
        if not name.startswith("swinViT.patch_embed.")
    ]
    if unexpected_keys or missing_non_patch_keys:
        raise RuntimeError(
            "Backbone-only checkpoint load failed. "
            f"Unexpected keys: {unexpected_keys}; missing non-patch keys: {missing_non_patch_keys}"
        )
    print(f"Loaded baseline backbone from {baseline_checkpoint}")
    print(f"Skipped baseline patch_embed keys: {skipped_patch_keys}")


def _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device):
    """Initialise every MSPE kernel from the pretrained baseline 2x2x2 patch_embed conv."""
    if not hasattr(mspe_embed, "patch_kernels"):
        print("Embedding has no patch_kernels; skipping warm-start.")
        return

    checkpoint = torch.load(baseline_checkpoint, weights_only=True)
    proj_w = checkpoint["swinViT.patch_embed.proj.weight"].to(device)  # [embed_dim, in_chans, 2, 2, 2]
    proj_b = checkpoint["swinViT.patch_embed.proj.bias"].to(device)
    base_size = tuple(proj_w.shape[2:])

    with torch.no_grad():
        for k in range(len(mspe_embed.patch_kernels)):
            kernel = mspe_embed.patch_kernels[k]
            k_size = tuple(kernel.weight.shape[2:])
            if k_size == base_size:
                kernel.weight.copy_(proj_w)                                      # exact (routing 2x2x2)
            else:
                kernel.weight.copy_(pi_resize(proj_w, list(k_size)).to(device))  # PI-resize to k_size
            if kernel.bias is not None:
                kernel.bias.copy_(proj_b)
        # warm-start the embedding norm affine as well
        if mspe_embed.norm is not None and "swinViT.patch_embed.norm.weight" in checkpoint:
            mspe_embed.norm.weight.copy_(checkpoint["swinViT.patch_embed.norm.weight"].to(device))
            mspe_embed.norm.bias.copy_(checkpoint["swinViT.patch_embed.norm.bias"].to(device))

    print(f"Warm-started {len(mspe_embed.patch_kernels)} MSPE kernels from baseline patch_embed "
          f"(exact copy where kernel size == {base_size}, PI-resize otherwise).")


def create_baseline_model(baseline_checkpoint, device):
    """Rebuild the stock-embed baseline and load the full checkpoint (for reference eval)."""
    model = monai.networks.nets.SwinUNETR(
        in_channels=1, out_channels=OUT_CHANNELS, spatial_dims=3,
        feature_size=FEATURE_SIZE, use_v2=True,
    ).to(device)
    model.swinViT.patch_embed = create_patch_embed_baseline().to(device)
    model.load_state_dict(torch.load(baseline_checkpoint, weights_only=True))
    model.eval()
    print(f"Loaded baseline checkpoint for evaluation from {baseline_checkpoint}")
    return model


def create_model_for_variant(variant_class, baseline_checkpoint, device):
    """SwinUNETR with a frozen baseline backbone + a trainable (warm-started) MSPE patch embed."""
    model = monai.networks.nets.SwinUNETR(
        in_channels=1, out_channels=OUT_CHANNELS, spatial_dims=3,
        feature_size=FEATURE_SIZE, use_v2=True,
    ).to(device)

    # Load only the frozen backbone -- skip the baseline patch_embed
    _load_baseline_backbone_only(model, baseline_checkpoint)

    # Build + swap the MSPE patch embedding
    mspe_embed = variant_class(
        patch_size=2, in_chans=1, embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm, spatial_dims=3,
        K=MSPE_K, resolutions=MSPE_RESOLUTIONS,
    ).to(device)

    if WARM_START:
        _warm_start_mspe_from_baseline(mspe_embed, baseline_checkpoint, device)

    model.swinViT.patch_embed = mspe_embed
    print(f"Swapped patch_embed to: {variant_class.__name__}")
    print(model.swinViT.patch_embed)

    # Freeze everything except the patch embedding
    for name, param in model.named_parameters():
        if "patch_embed" not in name:
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    frozen_params = total_params - trainable_params
    print(f" Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f" Frozen params:    {frozen_params:,} ({100 * frozen_params / total_params:.2f}%)")

    trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"  Trainable parameter groups ({len(trainable_names)}):")
    for n in trainable_names:
        print(f"   - {n}")

    return model

In [ ]:
# Evals helpers

post_pred = Compose([AsDiscrete(argmax=True, to_onehot=OUT_CHANNELS)])
post_label = Compose([AsDiscrete(to_onehot=OUT_CHANNELS)])

# HD95 spacing in mm (native grid after Spacingd)
HD95_SPACING = (0.75, 0.75, 1.0)


def _apply_endpoint_subtraction(outputs_list, labels_list, endpoints_list):
    """Zero the optional endpoint zones from both prediction and label one-hots."""
    for i in range(len(outputs_list)):
        endpoints_list[i][0, ...] = 0
        outputs_list[i] = torch.clamp(outputs_list[i] - endpoints_list[i], min=0)
        labels_list[i] = torch.clamp(labels_list[i] - endpoints_list[i], min=0)
    return outputs_list, labels_list


def _resized_hd95_spacing(native_shape, resized_shape):
    """Rescale mm spacing after an aspect-preserving resize so HD95 stays physical."""
    return tuple(
        HD95_SPACING[i] * (float(native_shape[i]) / float(resized_shape[i]))
        for i in range(len(native_shape))
    )


def run_validation(model, val_loader, dice_metric, subtract_endpoints=False):
    """Mean foreground Dice over the validation fold (used for model selection)."""
    model.eval()
    dice_metric.reset()
    with torch.no_grad():
        for val_data in val_loader:
            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)
            val_endpoints = val_data["endpoint"].to(device)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                val_outputs = model(val_inputs)

            outputs_list = [post_pred(i) for i in decollate_batch(val_outputs)]
            labels_list = [post_label(i) for i in decollate_batch(val_labels)]

            if subtract_endpoints:
                endpoints_list = [post_label(i) for i in decollate_batch(val_endpoints)]
                outputs_list, labels_list = _apply_endpoint_subtraction(outputs_list, labels_list, endpoints_list)

            dice_metric(y_pred=outputs_list, y=labels_list)

    metric = dice_metric.aggregate().item()
    dice_metric.reset()
    return metric


def _print_metric_table(title, dice_vals, hd95_vals):
    """Per-class Dice + HD95 table; returns (mean_dice, mean_hd95)."""
    num_classes = dice_vals.shape[0]
    print(title)
    print(f"{'Class':>8} | {'Dice':>8} | {'HD95':>10}")
    print("-" * 32)
    for c in range(num_classes):
        print(f"{c + 1:>8} | {dice_vals[c].item():>8.4f} | {hd95_vals[c].item():>10.2f}")
    mean_dice = dice_vals.nanmean().item()
    mean_hd95 = hd95_vals.nanmean().item()
    print("-" * 32)
    print(f"{'Mean':>8} | {mean_dice:>8.4f} | {mean_hd95:>10.2f}")
    return mean_dice, mean_hd95


def evaluate_at_resolution(model, val_loader, resolution=None, subtract_endpoints=False):
    """Per-class Dice + HD95 over the val fold at one effective resolution."""
    dice_per_class = DiceMetric(include_background=False, reduction="mean_batch")
    hd95_per_class = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean_batch")

    with torch.no_grad():
        for val_data in val_loader:
            val_inputs = val_data["image"].to(device)
            val_labels = val_data["label"].to(device)
            val_endpoints = val_data["endpoint"].to(device)

            native_shape = val_inputs.shape[2:]
            if resolution is None:
                spacing = HD95_SPACING
            else:
                target_size = get_aspect_preserving_target_size(val_inputs, resolution)
                val_inputs = img_resize(val_inputs, target_size)
                val_labels = label_resize(val_labels, target_size)
                val_endpoints = label_resize(val_endpoints, target_size)
                spacing = _resized_hd95_spacing(native_shape, target_size)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                val_outputs = model(val_inputs)

            outputs_list = [post_pred(i) for i in decollate_batch(val_outputs)]
            labels_list = [post_label(i) for i in decollate_batch(val_labels)]

            if subtract_endpoints:
                endpoints_list = [post_label(i) for i in decollate_batch(val_endpoints)]
                outputs_list, labels_list = _apply_endpoint_subtraction(outputs_list, labels_list, endpoints_list)

            outputs_cpu = [v.cpu() for v in outputs_list]
            labels_cpu = [v.cpu() for v in labels_list]
            dice_per_class(y_pred=outputs_cpu, y=labels_cpu)
            hd95_per_class(y_pred=outputs_cpu, y=labels_cpu, spacing=spacing)

    dice_vals = dice_per_class.aggregate()
    hd95_vals = hd95_per_class.aggregate()
    dice_per_class.reset()
    hd95_per_class.reset()

    tag = "native" if resolution is None else f"eff res {resolution}"
    mean_dice, mean_hd95 = _print_metric_table(f"\n[{tag}]  per-class Dice / HD95 (mm)", dice_vals, hd95_vals)
    return {
        "mean_dice": mean_dice, "mean_hd95": mean_hd95,
        "dice_vals": dice_vals.cpu(), "hd95_vals": hd95_vals.cpu(),
    }


def evaluate_all_resolutions(model, exp_name, val_loader, checkpoint_path, subtract_endpoints=False):
    """Native + per-resolution (MSPE_RESOLUTIONS) Dice/HD95 for the best checkpoint."""
    print(f"\n{'=' * 40}\nFINAL EVALUATION: {exp_name}\n{'=' * 40}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()

    native = evaluate_at_resolution(model, val_loader, resolution=None, subtract_endpoints=subtract_endpoints)
    multi_res = {}
    for r in MSPE_RESOLUTIONS:
        multi_res[r] = evaluate_at_resolution(model, val_loader, resolution=r, subtract_endpoints=subtract_endpoints)

    if wandb.run is not None:
        log_data = {"final/mean_dice": native["mean_dice"], "final/mean_hd95": native["mean_hd95"]}
        for c in range(native["dice_vals"].shape[0]):
            log_data[f"final/class_{c + 1}_dice"] = native["dice_vals"][c].item()
            log_data[f"final/class_{c + 1}_hd95"] = native["hd95_vals"][c].item()
        for r, res in multi_res.items():
            log_data[f"final_res_{r}/mean_dice"] = res["mean_dice"]
            log_data[f"final_res_{r}/mean_hd95"] = res["mean_hd95"]
            for c in range(res["dice_vals"].shape[0]):
                log_data[f"final_res_{r}/class_{c + 1}_dice"] = res["dice_vals"][c].item()
                log_data[f"final_res_{r}/class_{c + 1}_hd95"] = res["hd95_vals"][c].item()
        wandb.log(log_data)

    return {"native": native, "multi_res": multi_res}

In [ ]:
# MSPE kernel drift analysis

def analyze_kernel_drift(initial_state, model, exp_name):
    patch_embed = model.swinViT.patch_embed
    if not hasattr(patch_embed, "patch_kernels"):
        print(f"{exp_name}: no MSPE kernels; skipping drift analysis.")
        return None

    final_state = patch_embed.state_dict()
    metrics_l2, metrics_cos, metrics_max = [], [], []

    print(f"\nKernel drift analysis: {exp_name}")
    print("Kernel | L2 Dist  | Cosine Sim | Max Diff")
    print("-" * 44)
    for k in range(len(patch_embed.patch_kernels)):
        key = f"patch_kernels.{k}.weight"
        if key not in initial_state or key not in final_state:
            raise KeyError(f"Missing MSPE weight key for drift analysis: {key}")
        w_init = initial_state[key].detach().flatten().cpu()
        w_final = final_state[key].detach().flatten().cpu()
        l2 = torch.norm(w_final - w_init, p=2).item()
        cos = F.cosine_similarity(w_final.unsqueeze(0), w_init.unsqueeze(0)).item()
        mx = torch.max(torch.abs(w_final - w_init)).item()
        metrics_l2.append(l2); metrics_cos.append(cos); metrics_max.append(mx)
        print(f"{k:>6} | {l2:>8.4f} | {cos:>10.4f} | {mx:>8.4f}")

    drift = {"l2": metrics_l2, "cosine": metrics_cos, "max_diff": metrics_max}

    if wandb.run is not None:
        log_data = {}
        for k, (l2, cos, mx) in enumerate(zip(metrics_l2, metrics_cos, metrics_max)):
            log_data[f"kernel_drift/kernel_{k}_l2"] = l2
            log_data[f"kernel_drift/kernel_{k}_cosine"] = cos
            log_data[f"kernel_drift/kernel_{k}_max_diff"] = mx
        wandb.log(log_data)

    x = np.arange(len(metrics_l2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(x, metrics_l2); axes[0].set_title("L2 distance"); axes[0].set_xlabel("Kernel")
    axes[1].bar(x, metrics_cos); axes[1].set_title("Cosine similarity"); axes[1].set_xlabel("Kernel")
    axes[2].bar(x, metrics_max); axes[2].set_title("Max absolute diff"); axes[2].set_xlabel("Kernel")
    fig.suptitle(f"MSPE kernel drift: {exp_name}")
    plt.tight_layout()
    # Persist under root_dir (Drive) so the drift figure survives runtime.unassign()
    drift_path = os.path.join(root_dir, f"kernel_drift_{exp_name}_fold{fold_number}_{AUG_SHORT}.png")
    fig.savefig(drift_path, dpi=150, bbox_inches="tight")
    if wandb.run is not None:
        wandb.log({"kernel_drift/plot": wandb.Image(fig)})
    plt.show()
    return drift

In [ ]:
# Training loop (frozen backbone, only the MSPE patch_embed trains)

# Memory-efficient MSPE step for 3D
def mspe_swin_train_step_accum(
    model, img, label, loss_fn, scaler,
    lam=1.0, amp_dtype=AMP_DTYPE, device_type=device.type,
):
    patch_embed = model.swinViT.patch_embed
    hw_list = patch_embed.sample_resolutions()  # K resolutions, one per kernel
    K = len(hw_list)
    norm = K + 1
    running_loss = 0.0

    def _forward_backward(fwd_img, fwd_label, func_idx, weight):
        nonlocal running_loss
        with torch.autocast(device_type=device_type, dtype=amp_dtype):
            if func_idx is None:
                logits = model(fwd_img)  
            else:
                logits = mspe_swin_forward(model, fwd_img, func_idx=func_idx)
            loss = weight * loss_fn(logits, fwd_label) / norm
        scaler.scale(loss).backward()
        running_loss += loss.item()

    # K resolution-specific forwards, each back-propagated immediately
    for k, eff_target_k in enumerate(hw_list):
        target_size = get_aspect_preserving_target_size(img, eff_target_k)
        img_k = img_resize(img, target_size)
        label_k = label_resize(label, target_size)
        _forward_backward(img_k, label_k, func_idx=k, weight=1.0)
        del img_k, label_k

    # Native-resolution forward 
    _forward_backward(img, label, func_idx=None, weight=lam)
    return running_loss


def train_variant(model, train_loader, val_loader, exp_config):
    exp_name = exp_config["name"]
    checkpoint_path = exp_config["checkpoint_path"]

    loss_function = TverskyLoss(to_onehot_y=True, softmax=True, alpha=LOSS_ALPHA, beta=LOSS_BETA)
    # Optimize ONLY the trainable (patch_embed) parameters
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=WD,
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    scaler = GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    dice_metric = DiceMetric(include_background=False, reduction="mean")

    best_metric = -1
    best_metric_epoch = -1
    epoch_loss_values, metric_values = [], []
    epochs_no_improve = 0
    completed_epochs = 0
    total_start = time.time()

    for epoch in range(MAX_EPOCHS):
        epoch_start = time.time()
        print("-" * 10)
        print(f"{exp_name}: epoch {epoch + 1}/{MAX_EPOCHS}")
        model.train()
        epoch_loss = 0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs = batch_data["image"].to(device)
            labels = batch_data["label"].to(device)

            optimizer.zero_grad()
            # MSPE step with per forward backward
            step_loss = mspe_swin_train_step_accum(
                model, inputs, labels, loss_function, scaler, lam=1.0,
            )
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += step_loss
            if wandb.run is not None:
                wandb.log({"train/step_loss": step_loss, "train/amp_scale": scaler.get_scale()})
            print(f"{step}/{len(train_ds) // train_loader.batch_size}, train_loss: {step_loss:.4f}")

        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        completed_epochs = epoch + 1
        epoch_time_sec = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
        print(f"time per epoch: {epoch_time_sec:.2f} s")

        if wandb.run is not None:
            wandb.log({
                "train/epoch_loss": epoch_loss, "lr": current_lr,
                "epoch": epoch + 1, "train/time_per_epoch": epoch_time_sec,
            })

        if (epoch + 1) % VAL_INTERVAL == 0:
            metric = run_validation(model, val_loader, dice_metric, subtract_endpoints=SUBTRACT_ENDPOINTS)
            metric_values.append(metric)

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_path)
                print("saved new best metric model")
                print(f"current epoch: {epoch + 1} current mean dice: {metric:.3f}\n"
                      f"best mean dice: {best_metric:.3f} at epoch: {best_metric_epoch}")
            else:
                epochs_no_improve += VAL_INTERVAL

            if wandb.run is not None:
                wandb.log({"Dice": metric, "epoch": epoch + 1})

            if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch + 1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs).")
                if wandb.run is not None:
                    wandb.log({"early_stop_epoch": epoch + 1})
                break

        scheduler.step()

    total_time = time.time() - total_start
    avg_epoch_time = total_time / max(completed_epochs, 1)
    print(f"total epochs: {completed_epochs}, total training time: {total_time / 60:.2f} min, "
          f"average time per epoch: {avg_epoch_time:.2f} s")
    print(f"train completed, best Dice: {best_metric:.4f} at epoch: {best_metric_epoch}")
    return best_metric, best_metric_epoch, epoch_loss_values, metric_values

In [ ]:
# Baseline checkpoint eval (frozen reference) -- native + multi-resolution

login_wandb()
wandb.init(
    project=WANDB_PROJECT,
    name=f"BASELINE_CHECKPOINT_EVAL_{FOLD_TAG}_{AUG_TAG}",
    tags=[DATASET_TAG, FOLD_TAG, "baseline_eval"],
    config={
        "checkpoint": BASELINE_CHECKPOINT,
        "feature_size": FEATURE_SIZE,
        "out_channels": OUT_CHANNELS,
        "eval_resolutions": MSPE_RESOLUTIONS,
        "eval_only": True,
        "use_v2": True,
        "fold_number": fold_number,
    },
)

baseline_model = create_baseline_model(BASELINE_CHECKPOINT, device)

baseline_results = evaluate_all_resolutions(
    baseline_model, "BASELINE", val_loader, BASELINE_CHECKPOINT,
    subtract_endpoints=SUBTRACT_ENDPOINTS,
)

if wandb.run is not None:
    wandb.finish()
del baseline_model
torch.cuda.empty_cache()
print("Baseline checkpoint evaluation complete.")

In [ ]:
# Fine-tuning experiment loop

login_wandb()
assert "baseline_results" in globals(), (
    "Run the baseline checkpoint eval cell before running the experiments loop."
)

all_final_results = {"BASELINE": baseline_results}
all_training_curves = {}
all_drift = {}

for exp_idx, exp in enumerate(EXPERIMENTS):
    set_determinism(seed=0)
    exp_name = exp["name"]
    print(f"\n{'=' * 60}\nEXPERIMENT {exp_idx + 1}/{len(EXPERIMENTS)}: {exp['run_name']}\n{exp['description']}\n{'=' * 60}\n")

    wandb.init(
        project=WANDB_PROJECT,
        name=exp["run_name"],
        save_code=True,
        group=exp_name,
        tags=[DATASET_TAG, FOLD_TAG],
        config={
            "max_epochs": MAX_EPOCHS,
            "val_interval": VAL_INTERVAL,
            "batch_size": batch_size,
            "feature_size": FEATURE_SIZE,
            "out_channels": OUT_CHANNELS,
            "training_strategy": "frozen_backbone_mspe_only",
            "inference_strategy": "full_volume",
            "mspe_variant": exp["mspe_class"].__name__,
            "mspe_resolutions": MSPE_RESOLUTIONS,
            "mspe_K": MSPE_K,
            "eval_resolutions": MSPE_RESOLUTIONS,
            "frozen_backbone": True,
            "warm_start_kernels": WARM_START,
            "baseline_checkpoint": BASELINE_CHECKPOINT,
            "optional_zones_substraction": SUBTRACT_ENDPOINTS,
            "amp_dtype": str(AMP_DTYPE),
            "learning_rate": LR,
            "weight_decay": WD,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "loss_type": "TverskyLoss",
            "loss_alpha": LOSS_ALPHA,
            "loss_beta": LOSS_BETA,
            "fold_number": fold_number,
            "total_fold_number": total_fold_number,
            "augmentation": AUGMENT,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        },
    )

    # Frozen-backbone model with a warm-started MSPE patch embedding
    model = create_model_for_variant(exp["mspe_class"], BASELINE_CHECKPOINT, device)

    # Capture MSPE state AFTER warm-start so drift is measured from the seeded kernels
    initial_mspe_state = copy.deepcopy(model.swinViT.patch_embed.state_dict())

    best_metric, best_metric_epoch, epoch_losses, metric_vals = train_variant(model, train_loader, val_loader, exp)
    all_training_curves[exp_name] = {
        "epoch_losses": epoch_losses,
        "metric_values": metric_vals,
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
        "checkpoint_path": exp["checkpoint_path"],
    }

    all_final_results[exp_name] = evaluate_all_resolutions(
        model, exp_name, val_loader, exp["checkpoint_path"], subtract_endpoints=SUBTRACT_ENDPOINTS,
    )

    all_drift[exp_name] = analyze_kernel_drift(initial_mspe_state, model, exp_name)

    if wandb.run is not None:
        wandb.finish()
    del model
    torch.cuda.empty_cache()
    print(f"\n--- Completed {exp_name}")

print("\n" + "=" * 60)
print("ALL FINE-TUNING EXPERIMENTS COMPLETED")
print("=" * 60)

In [ ]:
# Summary tables and training curves

print(f"\n{'=' * 72}\nRESULTS SUMMARY (fold {fold_number})\n{'=' * 72}")
print(f"{'Condition':<20} | {'Best Dice':>10} | {'Best Epoch':>10} | {'Final Loss':>10} | {'Native HD95':>12}")
print("-" * 74)
for exp_name, curves in all_training_curves.items():
    final_loss = curves["epoch_losses"][-1] if curves["epoch_losses"] else float("nan")
    native = all_final_results.get(exp_name, {}).get("native", {})
    mean_hd95 = native.get("mean_hd95", float("nan"))
    print(f"{exp_name:<20} | {curves['best_metric']:>10.4f} | {curves['best_metric_epoch']:>10d} | "
          f"{final_loss:>10.4f} | {mean_hd95:>12.2f}")

# Resolution-robustness grids: native + each resized effective resolution
res_cols = ["Native"] + [str(r) for r in MSPE_RESOLUTIONS]
grid_header = f"{'Condition':<20} | " + " | ".join(f"{c:>8}" for c in res_cols) + " |"
grid_sep = "-" * len(grid_header)

print("\nMean Dice vs effective resolution:")
print(grid_sep); print(grid_header); print(grid_sep)
for exp_name, results in all_final_results.items():
    row = f"{exp_name:<20} | {results['native']['mean_dice']:>8.4f}"
    for r in MSPE_RESOLUTIONS:
        row += f" | {results['multi_res'][r]['mean_dice']:>8.4f}"
    print(row + " |")
print(grid_sep)

print("\nMean HD95 (mm) vs effective resolution:")
print(grid_sep); print(grid_header); print(grid_sep)
for exp_name, results in all_final_results.items():
    row = f"{exp_name:<20} | {results['native']['mean_hd95']:>8.2f}"
    for r in MSPE_RESOLUTIONS:
        row += f" | {results['multi_res'][r]['mean_hd95']:>8.2f}"
    print(row + " |")
print(grid_sep)

if all_drift:
    print("\nMSPE Kernel Drift Summary:")
    print(f"{'Condition':<20} | {'Kernel':>6} | {'L2':>8} | {'Cosine':>10} | {'Max Diff':>8}")
    print("-" * 62)
    for exp_name, drift in all_drift.items():
        if drift is None:
            continue
        for k, (l2, cos, mx) in enumerate(zip(drift["l2"], drift["cosine"], drift["max_diff"])):
            print(f"{exp_name:<20} | {k:>6d} | {l2:>8.4f} | {cos:>10.4f} | {mx:>8.4f}")

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
ax = axes[0]
for exp_name, curves in all_training_curves.items():
    epochs = list(range(1, len(curves["epoch_losses"]) + 1))
    ax.plot(epochs, curves["epoch_losses"], label=exp_name)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for exp_name, curves in all_training_curves.items():
    val_epochs = [VAL_INTERVAL * (i + 1) for i in range(len(curves["metric_values"]))]
    ax.plot(val_epochs, curves["metric_values"], label=exp_name, marker="o", markersize=3)
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation Dice"); ax.set_title("Validation Mean Dice")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
res_x = list(range(len(res_cols)))
for exp_name, results in all_final_results.items():
    dice_curve = [results["native"]["mean_dice"]] + [results["multi_res"][r]["mean_dice"] for r in MSPE_RESOLUTIONS]
    ax.plot(res_x, dice_curve, label=exp_name, marker="o", markersize=4)
ax.set_xticks(res_x); ax.set_xticklabels(res_cols, rotation=45)
ax.set_xlabel("Effective resolution"); ax.set_ylabel("Mean Dice"); ax.set_title("Resolution robustness")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Qualitative check: best fine-tuned checkpoints on the validation fold

for exp in EXPERIMENTS:
    ckpt = exp["checkpoint_path"]
    print(f"Visualizing {exp['name']} from {ckpt}")
    model_viz = create_model_for_variant(exp["mspe_class"], BASELINE_CHECKPOINT, device)
    model_viz.load_state_dict(torch.load(ckpt, weights_only=True))
    model_viz.eval()

    with torch.no_grad():
        for i, val_data in enumerate(val_loader):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                val_outputs = model_viz(val_data["image"].to(device))

            D = min(80, val_data["image"].shape[-1] - 1)
            fig = plt.figure("check", (18, 6))
            plt.subplot(1, 3, 1); plt.title(f"image {i}")
            plt.imshow(val_data["image"][0, 0, :, :, D].cpu(), cmap="gray")
            plt.subplot(1, 3, 2); plt.title(f"label {i}")
            plt.imshow(val_data["label"][0, 0, :, :, D].cpu())
            plt.subplot(1, 3, 3); plt.title(f"output {i}")
            plt.imshow(torch.argmax(val_outputs, dim=1).detach().cpu()[0, :, :, D])

            # Persist under root_dir (Drive) so figures survive runtime.unassign()
            viz_path = os.path.join(root_dir, f"viz_{exp['run_name']}_case{i}.png")
            fig.savefig(viz_path, dpi=150, bbox_inches="tight")
            plt.show()
            if i == 2:
                break

    del model_viz
    torch.cuda.empty_cache()

In [ ]:
# Disconnect the Colab runtime 
runtime.unassign()